# Module 2 — Calcul nutritionnel et détection des déséquilibres

Estimation des apports nutritionnels à partir des aliments détectés.
Dataset : Nutritional Facts for most common foods (Kaggle, 300+ aliments)

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/nutrition/nutrients_csvfile.csv")

print("Shape:", df.shape)
print("\nColonnes:", df.columns.tolist())
print("\nAperçu:")
df.head()

Shape: (335, 10)

Colonnes: ['Food', 'Measure', 'Grams', 'Calories', 'Protein', 'Fat', 'Sat.Fat', 'Fiber', 'Carbs', 'Category']

Aperçu:


,Food,Measure,Grams,Calories,Protein,Fat,Sat.Fat,Fiber,Carbs,Category
0,Cows' milk,1 qt.,976,660,32,40,36,0,48,Dairy products
1,Milk skim,1 qt.,984,360,36,t,t,0,52,Dairy products
2,Buttermilk,1 cup,246,127,9,5,4,0,13,Dairy products
3,"Evaporated, undiluted",1 cup,252,345,16,20,18,0,24,Dairy products
4,Fortified milk,6 cups,"1,419","1,373",89,42,23,1.4,119,Dairy products


In [4]:
# Nettoyage
print("Valeurs manquantes:")
print(df.isnull().sum())
print("\nValeurs uniques 'Measure':", df['Measure'].nunique())
print("\nCatégories:", df['Category'].unique())

Valeurs manquantes:
Food        0
Measure     0
Grams       0
Calories    1
Protein     0
Fat         0
Sat.Fat     2
Fiber       0
Carbs       0
Category    0
dtype: int64

Valeurs uniques 'Measure': 61

Catégories: <ArrowStringArray>
[                  'Dairy products',          'Fats, Oils, Shortenings',
                    'Meat, Poultry',                    'Fish, Seafood',
                   'Vegetables A-E',                   'Vegetables F-P',
                   'Vegetables R-Z',                       'Fruits A-F',
                       'Fruits G-P',                       'Fruits R-Z',
 'Breads, cereals, fastfood,grains',                            'Soups',
                 'Desserts, sweets',                    'Jams, Jellies',
                   'Seeds and Nuts',        'Drinks,Alcohol, Beverages']
Length: 16, dtype: str


In [5]:
# Nettoyage des données
df_clean = df.copy()

# Remplacer 't' (trace) par 0
df_clean = df_clean.replace('t', 0)

# Convertir les colonnes numériques
numeric_cols = ['Grams', 'Calories', 'Protein', 'Fat', 'Sat.Fat', 'Fiber', 'Carbs']
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Supprimer les virgules dans les nombres (ex: 1,419)
for col in numeric_cols:
    if df_clean[col].dtype == object:
        df_clean[col] = df_clean[col].str.replace(',', '').astype(float)

# Remplir les valeurs manquantes avec la médiane
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())

print("Après nettoyage:")
print(df_clean.isnull().sum())
print("\nAperçu:")
df_clean.head()

Après nettoyage:
Food        0
Measure     0
Grams       0
Calories    0
Protein     0
Fat         0
Sat.Fat     0
Fiber       0
Carbs       0
Category    0
dtype: int64

Aperçu:


,Food,Measure,Grams,Calories,Protein,Fat,Sat.Fat,Fiber,Carbs,Category
0,Cows' milk,1 qt.,976.0,660.0,32,40.0,36.0,0.0,48.0,Dairy products
1,Milk skim,1 qt.,984.0,360.0,36,0.0,0.0,0.0,52.0,Dairy products
2,Buttermilk,1 cup,246.0,127.0,9,5.0,4.0,0.0,13.0,Dairy products
3,"Evaporated, undiluted",1 cup,252.0,345.0,16,20.0,18.0,0.0,24.0,Dairy products
4,Fortified milk,6 cups,108.0,130.0,89,42.0,23.0,1.4,119.0,Dairy products


In [6]:
def estimate_nutrition(detected_foods, portion="standard"):
    """
    Estime les apports nutritionnels à partir d'une liste d'aliments détectés.
    
    Args:
        detected_foods: liste de labels détectés par le Module 1
        portion: "standard" (100g par défaut)
    
    Returns:
        dict avec calories, macros et déséquilibres détectés
    """
    results = []
    
    for food_label in detected_foods:
        # Recherche approximative dans le dataset
        matches = df_clean[df_clean['Food'].str.lower().str.contains(food_label.lower(), na=False)]
        
        if not matches.empty:
            row = matches.iloc[0]
            grams = row['Grams'] if row['Grams'] > 0 else 100
            factor = 100 / grams  # Normalise à 100g
            
            results.append({
                "food": food_label,
                "found_as": row['Food'],
                "calories_per_100g": round(row['Calories'] * factor, 1),
                "protein_per_100g": round(row['Protein'] * factor, 1),
                "carbs_per_100g": round(row['Carbs'] * factor, 1),
                "fat_per_100g": round(row['Fat'] * factor, 1),
                "fiber_per_100g": round(row['Fiber'] * factor, 1),
            })
        else:
            results.append({
                "food": food_label,
                "found_as": None,
                "calories_per_100g": None,
                "protein_per_100g": None,
                "carbs_per_100g": None,
                "fat_per_100g": None,
                "fiber_per_100g": None,
            })
    
    # Calcul des totaux
    found = [r for r in results if r['calories_per_100g'] is not None]
    
    if not found:
        return {"error": "Aucun aliment reconnu dans la base nutritionnelle"}
    
    total_calories = sum(r['calories_per_100g'] for r in found)
    total_protein = sum(r['protein_per_100g'] for r in found)
    total_carbs = sum(r['carbs_per_100g'] for r in found)
    total_fat = sum(r['fat_per_100g'] for r in found)
    
    return {
        "detected_foods": results,
        "totals": {
            "calories": round(total_calories, 1),
            "protein_g": round(total_protein, 1),
            "carbs_g": round(total_carbs, 1),
            "fat_g": round(total_fat, 1),
        }
    }

print("Fonction estimate_nutrition() prête !")

Fonction estimate_nutrition() prête !


In [7]:
# Test avec des aliments détectés par le Module 1
test_foods = ["banana", "tomato", "orange"]

result = estimate_nutrition(test_foods)

print("=== Résultat estimate_nutrition ===\n")
for food in result["detected_foods"]:
    print(f"Aliment : {food['food']}")
    print(f"  Trouvé comme : {food['found_as']}")
    print(f"  Calories/100g : {food['calories_per_100g']}")
    print(f"  Protéines/100g : {food['protein_per_100g']}g")
    print(f"  Glucides/100g : {food['carbs_per_100g']}g")
    print(f"  Lipides/100g : {food['fat_per_100g']}g")
    print()

print("=== Totaux ===")
print(f"Calories : {result['totals']['calories']} kcal")
print(f"Protéines : {result['totals']['protein_g']}g")
print(f"Glucides : {result['totals']['carbs_g']}g")
print(f"Lipides : {result['totals']['fat_g']}g")

=== Résultat estimate_nutrition ===

Aliment : banana
  Trouvé comme : Banana
  Calories/100g : 56.7
  Protéines/100g : 0.7g
  Glucides/100g : 15.3g
  Lipides/100g : 0.0g

Aliment : tomato
  Trouvé comme : Tomatoes
  Calories/100g : 20.8
  Protéines/100g : 0.8g
  Glucides/100g : 3.8g
  Lipides/100g : 0.0g

Aliment : orange
  Trouvé comme : Oranges 3" diameter
  Calories/100g : 33.3
  Protéines/100g : 1.1g
  Glucides/100g : 8.9g
  Lipides/100g : 0.0g

=== Totaux ===
Calories : 110.8 kcal
Protéines : 2.6g
Glucides : 28.0g
Lipides : 0.0g


In [8]:
def detect_imbalances(totals, user_goal="equilibre"):
    """
    Détecte les déséquilibres nutritionnels selon l'objectif utilisateur.
    
    Args:
        totals: dict avec calories, protein_g, carbs_g, fat_g
        user_goal: "perte_de_poids", "prise_de_masse", "equilibre"
    
    Returns:
        liste de déséquilibres détectés
    """
    imbalances = []
    
    calories = totals["calories"]
    protein = totals["protein_g"]
    carbs = totals["carbs_g"]
    fat = totals["fat_g"]
    
    # Seuils selon l'objectif
    if user_goal == "perte_de_poids":
        if calories > 400:
            imbalances.append("⚠️ Repas trop calorique pour un objectif de perte de poids")
        if carbs > 50:
            imbalances.append("⚠️ Glucides élevés — à réduire pour la perte de poids")
        if fat > 15:
            imbalances.append("⚠️ Lipides élevés")
            
    elif user_goal == "prise_de_masse":
        if protein < 20:
            imbalances.append("⚠️ Apport en protéines insuffisant pour la prise de masse")
        if calories < 500:
            imbalances.append("⚠️ Apport calorique insuffisant pour la prise de masse")
            
    else:  # equilibre
        if protein < 10:
            imbalances.append("⚠️ Manque de protéines")
        if carbs > 60:
            imbalances.append("⚠️ Glucides élevés")
        if fat > 20:
            imbalances.append("⚠️ Lipides élevés")
    
    if not imbalances:
        imbalances.append("✅ Repas équilibré selon votre objectif")
    
    return imbalances

# Test
imbalances = detect_imbalances(result["totals"], user_goal="perte_de_poids")
print("=== Déséquilibres détectés ===")
for i in imbalances:
    print(i)

=== Déséquilibres détectés ===
✅ Repas équilibré selon votre objectif


In [9]:
# Test avec un repas plus riche
test_foods_2 = ["pasta", "pork"]

result_2 = estimate_nutrition(test_foods_2)
imbalances_2 = detect_imbalances(result_2["totals"], user_goal="perte_de_poids")

print("=== Totaux ===")
print(f"Calories : {result_2['totals']['calories']} kcal")
print(f"Protéines : {result_2['totals']['protein_g']}g")
print(f"Glucides : {result_2['totals']['carbs_g']}g")
print(f"Lipides : {result_2['totals']['fat_g']}g")

print("\n=== Déséquilibres détectés ===")
for i in imbalances_2:
    print(i)

=== Totaux ===
Calories : 783.3 kcal
Protéines : 5.0g
Glucides : 0.0g
Lipides : 91.7g

=== Déséquilibres détectés ===
⚠️ Repas trop calorique pour un objectif de perte de poids
⚠️ Lipides élevés


In [10]:
# Sauvegarder le dataset nettoyé
df_clean.to_csv("data/processed/nutrition_clean.csv", index=False)
print("Dataset sauvegardé dans data/processed/nutrition_clean.csv")
print(f"Shape : {df_clean.shape}")

Dataset sauvegardé dans data/processed/nutrition_clean.csv
Shape : (335, 10)
